In [1]:
#@title beirutfreezone.come scraper
import requests
from IPython.display import display

base_url = "https://beirutfreezone.com/products.json"
all_products_data = []
page = 1
limit = 100

while True:
    params = {
        "page": page,
        "limit": limit
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()  # Raise an exception for HTTP errors
    data = response.json()

    products = data.get("products", [])

    if not products or len(products) < limit:
        # If no products or fewer than the limit, we've reached the last page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})
        break
    else:
        # Add all products from the current page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})

    page += 1

print(f"Found {len(all_products_data)} products.")
display(all_products_data)

Found 1890 products.


[{'id': 8473189810264,
  'handle': 'nivea-men-fresh-active-spray-deodorant-for-him-pack-of-3'},
 {'id': 8715546263640,
  'handle': 'dove-care-invisible-dry-48h-anti-perspirant-spray-deodorant-for-her-pack-of-3'},
 {'id': 8715546099800,
  'handle': 'emporio-armani-stronger-with-you-spices-eau-de-parfum-pour-homme-100ml'},
 {'id': 8715546132568,
  'handle': 'old-spice-restart-deodorant-stick-for-him-pack-of-3'},
 {'id': 8715546230872,
  'handle': 'nivea-fresh-natural-deodorant-stick-for-her-pack-of-3'},
 {'id': 8710299844696,
  'handle': 'azzaro-chrome-eau-de-parfum-pour-homme-50ml'},
 {'id': 8225809268824, 'handle': 'azzaro-chrome-parfum-pour-homme-100ml'},
 {'id': 8708864409688, 'handle': 'signal-medium-x-tra-clean-toothbrush'},
 {'id': 8708864507992, 'handle': 'oral-b-medium-5-way-clean-2-pcs'},
 {'id': 8708864376920,
  'handle': 'colgate-medium-360-floss-tip-toothbrush-3-pcs'},
 {'id': 8708864442456, 'handle': 'signal-soft-fighter-toothbrush-4-pcs'},
 {'id': 8698931118168,
  'handle'

In [3]:
#@title beirutfreezone.com products parser (with retry + delay + failure log)
import requests
import json
import csv
import time
from pathlib import Path
from typing import List, Dict

# ---------------- CONFIG ----------------
BASE_URL = "https://beirutfreezone.com/products/{}.js"
OUTPUT_JSON = "beirutfreezone.com_products.json"
OUTPUT_CSV = "beirutfreezone.com_products.csv"
FAILED_LOG = "failed_urls.json"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

DELAY_SECONDS = 1          # delay between requests
MAX_RETRIES = 3           # retry count
BACKOFF_FACTOR = 2        # exponential backoff


# ---------------- FETCH FUNCTION ----------------
def fetch_product(handle: str) -> Dict:
    url = BASE_URL.format(handle)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f"[RETRY {attempt}/{MAX_RETRIES}] {handle} failed: {e}")

            if attempt == MAX_RETRIES:
                return None

            sleep_time = DELAY_SECONDS * (BACKOFF_FACTOR ** (attempt - 1))
            time.sleep(sleep_time)

    return None


# ---------------- PARSE FUNCTION ----------------
def parse_product(product: Dict) -> List[Dict]:
    results = []

    product_data = {
        "product_id": product.get("id"),
        "title": product.get("title"),
        "handle": product.get("handle"),
        "description": product.get("description"),
        "created_at": product.get("created_at"),
        "vendor": product.get("vendor"),
        "type": product.get("type"),
        "price": product.get("price"),
        "compare_at_price": product.get("compare_at_price"),
        "available": product.get("available"),
    }

    variants = product.get("variants", [])

    if not variants:
        results.append(product_data)
        return results

    for variant in variants:
        row = product_data.copy()

        row.update({
            "variant_id": variant.get("id"),
            "variant_title": variant.get("title"),
            "variant_sku": variant.get("sku"),
            "variant_available": variant.get("available"),
            "variant_name": variant.get("name"),
            "variant_price": variant.get("price"),
            "variant_compare_at_price": variant.get("compare_at_price"),
            "variant_barcode": variant.get("barcode"),
            "variant_image": variant.get("featured_image", {}).get("src") if variant.get("featured_image") else None
        })

        results.append(row)

    return results


# ---------------- MAIN ----------------
def main():
    all_data = []
    failed_urls = []

    for item in all_products_data:
        handle = item["handle"]
        url = BASE_URL.format(handle)

        print(f"[INFO] Fetching {handle}")

        product_json = fetch_product(handle)

        if not product_json:
            failed_urls.append({
                "handle": handle,
                "url": url
            })
            continue

        parsed_rows = parse_product(product_json)
        all_data.extend(parsed_rows)

        # delay after each request
        time.sleep(DELAY_SECONDS)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    # Save CSV
    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_data)

    # Save failed URLs
    with open(FAILED_LOG, "w", encoding="utf-8") as f:
        json.dump(failed_urls, f, indent=2)

    print(f"[DONE] Saved {len(all_data)} rows")
    print(f"[FAILED] {len(failed_urls)} URLs logged to {FAILED_LOG}")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Fetching nivea-men-fresh-active-spray-deodorant-for-him-pack-of-3
[INFO] Fetching dove-care-invisible-dry-48h-anti-perspirant-spray-deodorant-for-her-pack-of-3
[INFO] Fetching emporio-armani-stronger-with-you-spices-eau-de-parfum-pour-homme-100ml
[INFO] Fetching old-spice-restart-deodorant-stick-for-him-pack-of-3
[INFO] Fetching nivea-fresh-natural-deodorant-stick-for-her-pack-of-3
[INFO] Fetching azzaro-chrome-eau-de-parfum-pour-homme-50ml
[INFO] Fetching azzaro-chrome-parfum-pour-homme-100ml
[INFO] Fetching signal-medium-x-tra-clean-toothbrush
[INFO] Fetching oral-b-medium-5-way-clean-2-pcs
[INFO] Fetching colgate-medium-360-floss-tip-toothbrush-3-pcs
[INFO] Fetching signal-soft-fighter-toothbrush-4-pcs
[INFO] Fetching prada-lhomme-eau-de-toilette-pour-homme-100ml
[INFO] Fetching gucci-guilty-mens-gift-set
[INFO] Fetching gucci-guilty-eau-de-parfum-pour-femme-90ml
[INFO] Fetching azzaro-wanted-eau-de-parfum-pour-homme-100ml
[INFO] Fetching hugo-boss-bottled-parfum-pour-homme-1